# Table 1：8 个数据选择算法的可审计复现协议

本 Notebook 将固定提交 `8cf757adc4c333dc1427d511f0de2f246d15ebac` 的 Table 1 复现配置整理为可执行检查。默认只做协议预检、命令预览和已有结果完整性分析，不自动启动正式训练。

目标规模：`5 datasets × 2 ratios × 8 methods × 5 trials = 400 results`。由于公开仓库没有作者 Table 1 对应的完整原始配置和选集文件，最终应称为 **fixed-commit auditable reproduction**，不能称为恢复了所有未公开原始运行细节。

## 方法配置

| Table 1 方法 | `--methods` | 关键配置 |
|---|---|---|
| Random | `random` | 每类独立无放回采样；不要改成 `random_proportional` |
| EL2N | `el2n_top` | 200 epoch dynamics；前 20 epoch 概率误差 L2 均值；28×28；每类取最高分 |
| Forgetting | `forgetting` | 200 epoch 中正确到错误的次数；每类取最高分 |
| EVA | `eva` | 两个窗口的逐样本 L2 误差方差之和；`eva-epochs=200`、窗口宽度 10、打分 28×28 |
| Facility | `facility` | UNI；per-class greedy facility location；不加 `--global` |
| FPS | `fps` | UNI；每类归一化 embedding 上的 farthest-point sampling |
| Herding | `herding` | UNI；迭代匹配类均值，不是一次性最近中心排序 |
| Ours / GraphCov | `graph_a2` | UNI；global graph；`k=50`；`H=2`；`A_sym + A_sym^2`；equal class quota |

GraphCov 命令必须同时包含 `--embeddings uni -k 50 --k-hops 2 --global`。只写 `-k 50` 会进入 per-class 分支，不是论文描述的 global 方法。

## 统一下游训练协议

所有方法选完子集后统一使用：ResNet-18 from scratch，224×224，epoch 模式，1000 epochs，batch size 256，SGD，learning rate 0.1，momentum 0.9，weight decay 5e-4，CosineAnnealingLR(T_max=1000)，无 augmentation、无 Nesterov，seeds 42–46。

主指标固定为最终 epoch 的 `balanced_accuracy`。`best_balanced_accuracy` 只能作为审计字段，不能在不同方法之间混用 final 和 best，也不能使用测试集峰值来反向改配置。

In [ ]:
from pathlib import Path
import json
import os
import shlex
import pandas as pd

# 修改为 graph-coverage-selection 仓库根目录；默认假设 Notebook 在 table1_reproduction 下。
PROJECT_ROOT = Path(os.environ.get('GRAPHCOV_ROOT', Path.cwd()))
OUT = PROJECT_ROOT / 'repro_table1_8cf757a_epoch1000_dyn28'
CACHE = PROJECT_ROOT / 'cache_table1_8cf757a_dyn28'
DATASETS = ['organsmnist', 'organamnist', 'pathmnist', 'tissuemnist', 'bloodmnist']
RATIOS = [0.02, 0.05]
METHODS = ['random', 'el2n_top', 'forgetting', 'eva', 'facility', 'fps', 'herding', 'graph_a2 (global)']
SEEDS = [42, 43, 44, 45, 46]

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUT =', OUT)
print('正式训练默认关闭；本 Notebook 只预检和分析已有结果。')

## HPC 路径与 Slurm 入口

正式 HPC 运行使用以下固定路径和环境；Notebook 只展示并校验配置，不直接提交作业。项目根目录是 `/project/PROJECT_ROOT/USER/graph_bench`，Python 3.11 环境是 `/project/PROJECT_ROOT/USER/graph_select/envs/graphcov-py311/bin/python`。GPU 选择阶段使用 `YOUR_GPU_PARTITION + YOUR_SLURM_QOS`，CPU 汇总使用 `cpu-amd9754 + qos-normal`。

In [ ]:
HPC_PROJECT_ROOT = Path('/project/PROJECT_ROOT/USER/graph_bench')
HPC_PYTHON = Path('/project/PROJECT_ROOT/USER/graph_select/envs/graphcov-py311/bin/python')
HPC_DATA_ROOT = Path.home() / '.medmnist'
HPC_LOG_ROOT = HPC_PROJECT_ROOT / 'table1_reproduction' / 'logs'
HPC_SLURM = {
    'selection': HPC_PROJECT_ROOT / 'table1_reproduction/slurm/job1_select_array.slurm',
    'downstream': HPC_PROJECT_ROOT / 'table1_reproduction/slurm/job2_downstream_array.slurm',
    'summary': HPC_PROJECT_ROOT / 'table1_reproduction/slurm/table1_summary.slurm',
}
HPC_SHELL_CHECKS = [
    f'cd {HPC_PROJECT_ROOT}',
    f'{HPC_PYTHON} -m unittest discover -s table1_reproduction/tests -v',
    f'bash -n {HPC_SLURM["selection"]}',
    f'bash -n {HPC_SLURM["downstream"]}',
    f'{HPC_PYTHON} -m py_compile table1_reproduction/experiments/job1_select.py table1_reproduction/experiments/job2_downstream.py table1_reproduction/experiments/summarize.py',
]
print('HPC project:', HPC_PROJECT_ROOT)
print('HPC Python:', HPC_PYTHON)
print('HPC selection Slurm:', HPC_SLURM['selection'])
print('HPC downstream Slurm:', HPC_SLURM['downstream'])
print('HPC summary:', HPC_SLURM['summary'])
print('\n'.join(HPC_SHELL_CHECKS))

In [ ]:
# 论文明确/源码确认/复现假设的边界记录
protocol = {
    'commit': '8cf757adc4c333dc1427d511f0de2f246d15ebac',
    'datasets': DATASETS,
    'ratios': RATIOS,
    'trials': 5,
    'seeds': SEEDS,
    'selection_embedding': 'UNI',
    'downstream': {
        'model': 'ResNet-18 from scratch',
        'size': 224,
        'epochs': 1000,
        'batch_size': 256,
        'optimizer': 'SGD',
        'lr': 0.1,
        'momentum': 0.9,
        'weight_decay': 5e-4,
        'scheduler': 'CosineAnnealingLR(T_max=1000)',
        'augmentation': False,
        'primary_metric': 'balanced_accuracy',
        'primary_endpoint': 'final epoch',
    },
    'assumptions': {
        'graph_a2_k': 50,
        'graph_a2_hops': 2,
        'facility_scope': 'per-class',
        'dynamics_size': 28,
    },
}
protocol

In [ ]:
# 每类预算：B=floor(floor(N*r)/C)，总选集大小为 B*C。
dataset_sizes = {
    'organsmnist': (13932, 11),
    'organamnist': (34561, 11),
    'pathmnist': (89996, 9),
    'tissuemnist': (165466, 8),
    'bloodmnist': (11959, 8),
}
budget_rows = []
for dataset, (n_train, n_classes) in dataset_sizes.items():
    row = {'dataset': dataset, 'n_train': n_train, 'classes': n_classes}
    for ratio in RATIOS:
        per_class = int(n_train * ratio) // n_classes
        row[f'{ratio:.0%}_per_class'] = per_class
        row[f'{ratio:.0%}_total'] = per_class * n_classes
    budget_rows.append(row)
pd.DataFrame(budget_rows)

## 三组正式命令

三个方法组应分开执行，避免 `--global` 意外作用于 Facility。下面只生成命令，不执行命令。正式运行前应确认输出目录不存在，保存 `pip freeze`、commit、CUDA、PyTorch、TorchVision、FAISS、timm 和 GPU 信息。

In [ ]:
common = [
    'python', '-m', 'graphcov.run',
    '--datasets', *DATASETS,
    '--ratios', '0.02', '0.05',
    '--trials', '5',
    '--training-paradigm', 'epoch', '--epochs', '1000',
    '--size', '224', '--batch-size', '256', '--lr', '0.1',
    '--momentum', '0.9', '--weight-decay', '0.0005',
    '--seed', '42', '--num-workers', '4',
    '--test-every-n-epochs', '10',
    '--output', str(OUT), '--cache', str(CACHE),
]
commands = {
    'A_dynamics': common + ['--methods', 'random', 'el2n_top', 'forgetting', 'eva', '--eva-epochs', '200', '--eva-window-size', '10', '--eva-size', '28'],
    'B_geometry': common + ['--methods', 'facility', 'fps', 'herding', '--embeddings', 'uni'],
    'C_graphcov_ours': common + ['--methods', 'graph_a2', '--embeddings', 'uni', '-k', '50', '--k-hops', '2', '--global'],
}
for name, command in commands.items():
    print(f'[{name}]\n  ' + shlex.join(command) + '\n')

## 已有 `results.csv` 的完整性检查

将 `RESULTS_CSV` 环境变量或下面的 `results_csv` 指向已有结果。检查会拒绝缺失/重复 trial、错误 seed 集合、GraphCov 非 global 行和 Facility global 行。

In [ ]:
results_csv = Path(os.environ.get('RESULTS_CSV', OUT / 'results.csv'))
if not results_csv.is_file():
    print('未找到 results.csv；跳过完整性检查：', results_csv)
else:
    df = pd.read_csv(results_csv)
    keys = ['dataset', 'ratio', 'method']
    expected = pd.MultiIndex.from_product([DATASETS, RATIOS, METHODS], names=keys)
    assert len(df) == 400, f'应有 400 条结果，实际为 {len(df)} 条'
    assert not df.duplicated(keys + ['seed']).any(), '存在重复 trial'
    counts = df.groupby(keys).size().reindex(expected, fill_value=0)
    assert (counts == 5).all(), counts[counts != 5]
    assert set(df['seed']) == set(SEEDS), sorted(df['seed'].unique())
    graph_rows = df[df['base_method'] == 'graph_a2']
    assert graph_rows['method'].eq('graph_a2 (global)').all(), 'GraphCov 存在非 global 行'
    facility_rows = df[df['base_method'] == 'facility']
    assert not facility_rows['method'].str.contains('global', regex=False).any(), 'Facility 被错误设置为 global'
    print('完整性检查通过：400 条结果，80 个单元，每个单元 5 个 trial。')
    display(df.groupby(keys)['balanced_accuracy'].agg(['mean', 'std']).head())

## 预算与证据边界

每类预算按源码取整：`B = floor(floor(N*r)/C)`。本协议将所有数据集的 GraphCov 统一为 `k=50, H=2, global`，Facility 保留 per-class，动态打分保留 28×28。这三点属于复现假设，不是作者逐格原始运行记录。

论文和源码没有同时提供 Table 1 每格的原始命令、选集索引、逐 trial CSV、完整环境锁定和汇总脚本。因此正式报告应写：

> Reproduction on commit `8cf757a` under a disclosed, unified Table 1 protocol.

而不应写成：

> Exact reproduction of every unpublished original Table 1 configuration.

## 审计入口

- CLI 参数和默认值：`graphcov/run/__main__.py`
- 方法注册、选择方向和图算法：`graphcov/run/selection.py`
- trial seed、预算、动态分数和结果写入：`graphcov/run/experiment.py`
- EVA、Forgetting 和 EL2N dynamics：`graphcov/run/eva.py`
- 下游训练和 balanced accuracy：`graphcov/run/evaluation.py`

Notebook 默认不提交训练作业；正式运行应通过经过审计的 shell/Slurm 脚本，并把命令、配置、环境和结果路径一并保存。